# Кредитный скоринг: от запроса бизнеса до датасета

Бизнес пришёл с фразой: «Просрочка выросла. Нужна модель надёжности клиента».

Ваша задача не обучить модель, а сделать всё, что должно случиться до обучения:
разобраться в данных, собрать target, проанализировать проблему и подготовить датасет.

## Данные

- Все данные задания лежат в Postgres, схема `public`, таблицы с префиксом `scoring_`. Дата выгрузки: **2025-05-31**.
- Таблицы без префикса `scoring_` к заданию не относятся, даже если выглядят похоже.
- В выгрузку попало всё подряд: устаревшие копии, бэкапы, служебные таблицы. Не каждая `scoring_`-таблица пригодна для работы. Решить, какие использовать, а какие забраковать, и обосновать это, часть задания.

In [1]:
# pip install psycopg2-binary pandas sqlalchemy
import os
from pathlib import Path
from urllib.parse import quote_plus
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

# Загружаем логин и пароль из локального файла .env (не попадает в Git)
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

DSN = (
    f"postgresql://{quote_plus(os.environ['DB_USER'])}:{quote_plus(os.environ['DB_PASSWORD'])}"
    f"@{os.getenv('DB_HOST', 'thomas.proxy.rlwy.net')}:{os.getenv('DB_PORT', '51432')}"
    f"/{os.getenv('DB_NAME', 'academy_db')}"
)
engine = create_engine(DSN)

def q(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print("Подключение к БД успешно настроено!")

Подключение к БД успешно настроено!


In [2]:
# Все таблицы задания
q("""
select tablename
from pg_tables
where schemaname = 'public' and tablename like 'scoring\_%'
order by 1
""")

<>:5: SyntaxWarning: invalid escape sequence '\_'
<>:5: SyntaxWarning: invalid escape sequence '\_'
C:\Users\toshp\AppData\Local\Temp\ipykernel_22416\3674523586.py:5: SyntaxWarning: invalid escape sequence '\_'
  where schemaname = 'public' and tablename like 'scoring\_%'


,tablename
0,scoring_applications
1,scoring_applications_full
2,scoring_bureau_inquiries
3,scoring_bureau_loans
4,scoring_bureau_snapshots
5,scoring_card_disputes
6,scoring_card_transactions
7,scoring_cards
8,scoring_clients_old
9,scoring_collection_actions


## Шаг 1. Постановка задачи

Ответьте письменно (в ячейке ниже) на пять вопросов из лекции:

1. Какое решение банка изменится (одобрение, лимит, ставка)?
2. Какое событие прогнозируем и в каком горизонте?
3. На какой популяции применяется решение?
4. Какие данные доступны в момент решения?
5. По какому критерию бизнес примет результат?

Если на вопрос нет ответа, запишите его как допущение.

### Ответы на вопросы постановки задачи (Application-скоринг)

1. **Какое решение банка изменится (одобрение, лимит, ставка)?**
   - **Основное решение:** бинарное решение об одобрении или отказе по кредитной заявке (`approve` / `reject`) на этапе рассмотрения заявки (`decision_date`) взамен устаревшей политики (`approved_by_old_policy` и `scoring_model_v1_scores`).
   - **Допущение по дополнительным рычагам:** для пограничных клиентов («серая зона» вероятности дефолта PD) скорбалл может использоваться для снижения одобряемого лимита (`issued_amount_uzs < requested_amount_uzs`) или риск-ориентированной надбавки к ставке (`annual_rate_pct`).

2. **Какое событие прогнозируем и в каком горизонте?**
   - **Целевое событие (`bad = 1`):** выход договора в серьёзную просрочку **DPD $\ge$ 60 дней** (Days Past Due 60+ по любому из плановых платежей) в течение **первых 6 месяцев (6 плановых платежей, `installment_no <= 6`)** с момента выдачи кредита.
   - **Почему выбран горизонт 6 месяцев и порог 60+ дней:**
     - Минимальный срок кредита в портфеле равен **6 месяцам** (`term_months` от 6 до 24 мес.), поэтому горизонт 6 месяцев уравнивает шансы всех договоров и не зависит от длительности кредита.
     - Проверка по базе платежей показывает, что **100% всех дефолтов DPD 60+ и DPD 90+ за всю историю портфеля впервые случаются именно на первых 6 платежах** (`installment_no` с 1 по 6). Увеличение горизонта до 12 месяцев не добавляет ни одного нового дефолта, но цензурирует (отсекает) свежие выдачи 2024 года.
     - Порог **DPD 60+** отсекает случайные технические задержки (при DPD 30+ часть клиентов ещё гасит долг), но даёт больше зрелых когорт, чем DPD 90+.

3. **На какой популяции применяется решение?**
   - **Популяция применения (inference):** все входящие кредитные заявки физических лиц во всех каналах (`branch`, `mobile_app`, `agent`), включая как действующих клиентов банка (ETB — *Existing-to-Bank*, есть в `scoring_customers`), так и новых клиентов с улицы (NTB — *New-to-Bank*, нет в `scoring_customers`).
   - **Популяция разработки (обучения):** исторически **одобренные и выданные** заявки (`final_decision = 'approved'`), по которым закрылось окно наблюдения (выдачи с июня 2023 по сентябрь 2024 включительно). Так как по отклонённым заявкам (`rejected`) истинный таргет неизвестен, возникает смещение выборки (*selection bias*), которое мы учитываем при анализе и валидации.

4. **Какие данные доступны в момент решения?**
   - Строго те данные, которые зафиксированы **на дату `decision_date` (или ранее)**:
     1. **Анкета заявки (`scoring_applications`):** запрошенная сумма, срок, заявленный доход, стаж, канал, цель кредита.
     2. **Профиль клиента (`scoring_customers`):** дата онбординга, регион, пол, возраст, сегмент (только если `onboarding_date <= decision_date`).
     3. **Кредитное бюро (`scoring_bureau_snapshots`, `scoring_bureau_inquiries`):** последний снимок БКИ со `snapshot_date <= decision_date` (*as-of join*) и история запросов в БКИ за 3–6 месяцев до `decision_date`.
     4. **Карточное поведение (`scoring_cards`, `scoring_card_transactions`, `scoring_card_disputes`):** обороты, зарплатные начисления (`salary_in`), снятия наличных (`cash_out`), переводы, доля отказов (`declined`) и диспуты, открытые строго до `decision_date`.

5. **По какому критерию бизнес примет результат?**
   - **Качество ранжирования (DS-метрики на out-of-time выборке):** ROC-AUC $\ge 0.75$ (или Gini $\ge 0.50$), прирост ROC-AUC и PR-AUC относительно старой модели `score_v1`, стабильность распределения скора во времени (PSI $< 0.10$).
   - **Бизнес-критерий (экономика решения):** снижение доли дефолтов (Bad Rate) в новых когортах выдачи с текущих ~22–28% до целевого уровня $\le 12\text{–}15\%$ при сохранении уровня одобрения (Approval Rate $\approx 60\%$) либо максимизация чистой прибыли портфеля (NPV) с учётом асимметричной цены ошибок: одобрение `bad`-клиента стоит банку в среднем **~8.68 млн сум** прямого убытка по телу, а отказ `good`-клиенту — **~2.35 млн сум** упущенного процентного дохода.

## Шаг 2. Разведка данных

Для каждой `scoring_`-таблицы ответьте: что за сущность, какой ключ, за какой период данные,
какие колонки доступны ДО момента решения, а какие появляются после.

Отдельно проверьте:

- сколько заявок одобрено и отклонено, что известно об отклонённых;
- есть ли договоры, по которым окно наблюдения ещё не закрылось;
- какие колонки выглядят полезными, но содержат информацию из будущего;
- какие таблицы дублируют друг друга и какая из версий полная и актуальная;
- для каких клиентов есть карточные транзакции, а для каких их нет и почему.

Результат: таблица «источник, ключ, момент появления, можно ли в фичи», включая забракованные таблицы с причиной.

In [3]:
# 2.1. Обзор всех 17 таблиц scoring_*: размер, ключи и диапазоны дат
overview_sql = """
SELECT 'scoring_applications' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT application_id) AS uniq_key,
       MIN(decision_date)::text AS min_date, MAX(decision_date)::text AS max_date FROM scoring_applications
UNION ALL
SELECT 'scoring_applications_full', COUNT(*), COUNT(DISTINCT application_id),
       MIN(decision_date)::text, MAX(decision_date)::text FROM scoring_applications_full
UNION ALL
SELECT 'scoring_contracts', COUNT(*), COUNT(DISTINCT contract_id),
       MIN(issue_date)::text, MAX(issue_date)::text FROM scoring_contracts
UNION ALL
SELECT 'scoring_payments', COUNT(*), COUNT(DISTINCT payment_id),
       MIN(due_date)::text, MAX(due_date)::text FROM scoring_payments
UNION ALL
SELECT 'scoring_payments_backup', COUNT(*), COUNT(DISTINCT payment_id),
       MIN(due_date)::text, MAX(due_date)::text FROM scoring_payments_backup
UNION ALL
SELECT 'scoring_customers', COUNT(*), COUNT(DISTINCT customer_id),
       MIN(onboarding_date)::text, MAX(onboarding_date)::text FROM scoring_customers
UNION ALL
SELECT 'scoring_clients_old', COUNT(*), COUNT(DISTINCT client_id),
       MIN(birthdate)::text, MAX(birthdate)::text FROM scoring_clients_old
UNION ALL
SELECT 'scoring_bureau_snapshots', COUNT(*), COUNT(DISTINCT (client_id, snapshot_date)),
       MIN(snapshot_date)::text, MAX(snapshot_date)::text FROM scoring_bureau_snapshots
UNION ALL
SELECT 'scoring_bureau_inquiries', COUNT(*), COUNT(DISTINCT inquiry_id),
       MIN(inquiry_date)::text, MAX(inquiry_date)::text FROM scoring_bureau_inquiries
UNION ALL
SELECT 'scoring_bureau_loans', COUNT(*), COUNT(DISTINCT bureau_loan_id),
       MIN(open_date)::text, MAX(open_date)::text FROM scoring_bureau_loans
UNION ALL
SELECT 'scoring_cards', COUNT(*), COUNT(DISTINCT card_id),
       MIN(issue_date)::text, MAX(issue_date)::text FROM scoring_cards
UNION ALL
SELECT 'scoring_card_transactions', COUNT(*), COUNT(DISTINCT txn_id),
       MIN(txn_ts::date)::text, MAX(txn_ts::date)::text FROM scoring_card_transactions
UNION ALL
SELECT 'scoring_txn_sample', COUNT(*), COUNT(DISTINCT transaction_id),
       MIN(txn_ts::date)::text, MAX(txn_ts::date)::text FROM scoring_txn_sample
UNION ALL
SELECT 'scoring_card_disputes', COUNT(*), COUNT(DISTINCT dispute_id),
       MIN(opened_date)::text, MAX(opened_date)::text FROM scoring_card_disputes
UNION ALL
SELECT 'scoring_collection_actions', COUNT(*), COUNT(DISTINCT (contract_id, action_date, action_type)),
       MIN(action_date)::text, MAX(action_date)::text FROM scoring_collection_actions
UNION ALL
SELECT 'scoring_model_v1_scores', COUNT(*), COUNT(DISTINCT application_id),
       NULL, NULL FROM scoring_model_v1_scores
UNION ALL
SELECT 'scoring_temp', COUNT(*), COUNT(DISTINCT id),
       NULL, NULL FROM scoring_temp
"""
display(q(overview_sql))

,table_name,rows,uniq_key,min_date,max_date
0,scoring_temp,0,0,NaN,NaN
1,scoring_card_disputes,2467,2467,2023-01-06,2025-06-25
2,scoring_customers,8000,8000,2017-01-01,2022-12-31
3,scoring_txn_sample,5000,5000,2023-01-01,2023-12-28
4,scoring_contracts,11277,11277,2023-06-02,2025-05-05
5,scoring_clients_old,6000,6000,1965-01-01,2004-12-30
6,scoring_cards,12614,12614,2017-01-03,2024-12-31
7,scoring_collection_actions,9135,9135,2023-08-04,2025-11-30
8,scoring_applications,18000,18000,2023-06-01,2025-04-30
9,scoring_applications_full,11277,11277,2023-06-01,2025-04-30


In [4]:
# 2.2. Проверка 1: Одобренные и отклонённые заявки, связь со старой политикой и договорами
print("--- 1. Статистика решений по заявкам (scoring_applications) ---")
display(q("""
SELECT 
    a.final_decision,
    a.approved_by_old_policy,
    COUNT(*) AS applications_cnt,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS share_pct,
    COUNT(c.contract_id) AS contracts_cnt,
    COUNT(cu.customer_id) AS in_customers_cnt,
    COUNT(*) FILTER (WHERE EXISTS (
        SELECT 1 FROM scoring_bureau_snapshots b
        WHERE b.client_id = a.client_id AND b.snapshot_date <= a.decision_date
    )) AS has_bureau_snapshot_cnt
FROM scoring_applications a
LEFT JOIN scoring_contracts c ON a.application_id = c.application_id
LEFT JOIN scoring_customers cu ON a.client_id = cu.customer_id
GROUP BY 1, 2
ORDER BY 1, 2
"""))

# 2.3. Проверка 2: Дублирующиеся и устаревшие таблицы
print("--- 2. Проверка дубликатов: scoring_clients_old vs scoring_customers ---")
display(q("""
SELECT 
    (SELECT COUNT(*) FROM scoring_customers) AS customers_rows,
    (SELECT COUNT(*) FROM scoring_clients_old) AS clients_old_rows,
    COUNT(*) AS matched_in_customers
FROM scoring_clients_old o
JOIN scoring_customers c ON o.client_id = c.customer_id
"""))

print("--- 3. Проверка бэкапа платежей: scoring_payments_backup vs scoring_payments ---")
display(q("""
SELECT 
    (SELECT COUNT(*) FROM scoring_payments) AS payments_rows,
    COUNT(*) AS backup_rows,
    COUNT(DISTINCT b.payment_id) AS backup_uniq_payment_id,
    COUNT(*) - COUNT(DISTINCT b.payment_id) AS duplicate_rows_in_backup,
    COUNT(p.payment_id) AS matched_in_main_payments,
    MAX(b.due_date) AS backup_max_due_date,
    (SELECT MAX(due_date) FROM scoring_payments) AS main_max_due_date
FROM scoring_payments_backup b
LEFT JOIN scoring_payments p ON b.payment_id = p.payment_id
"""))

# 2.4. Проверка 3: Для каких клиентов есть карточные транзакции, а для каких нет и почему
print("--- 4. Наличие карт и транзакций в разрезе сегментов (scoring_customers) ---")
display(q("""
WITH app_card_check AS (
    SELECT 
        a.application_id,
        a.client_id,
        a.decision_date,
        cu.customer_segment,
        COUNT(DISTINCT ca.card_id) AS cards_cnt,
        COUNT(DISTINCT t.txn_id) AS txns_total,
        COUNT(DISTINCT t.txn_id) FILTER (WHERE t.txn_ts < a.decision_date) AS txns_before_decision
    FROM scoring_applications a
    LEFT JOIN scoring_customers cu ON a.client_id = cu.customer_id
    LEFT JOIN scoring_cards ca ON a.client_id = ca.customer_id
    LEFT JOIN scoring_card_transactions t ON a.client_id = t.client_id
    GROUP BY 1, 2, 3, 4
)
SELECT 
    COALESCE(customer_segment, 'NTB (нет в scoring_customers)') AS client_group,
    COUNT(*) AS applications_cnt,
    COUNT(DISTINCT client_id) AS unique_clients,
    COUNT(*) FILTER (WHERE cards_cnt > 0) AS apps_with_cards,
    COUNT(*) FILTER (WHERE txns_total > 0) AS apps_with_any_txns,
    COUNT(*) FILTER (WHERE txns_before_decision > 0) AS apps_with_txns_before_decision
FROM app_card_check
GROUP BY 1
ORDER BY applications_cnt DESC
"""))

--- 1. Статистика решений по заявкам (scoring_applications) ---


,final_decision,approved_by_old_policy,applications_cnt,share_pct,contracts_cnt,in_customers_cnt,has_bureau_snapshot_cnt
0,approved,False,362,2.01,362,149,211
1,approved,True,10915,60.64,10915,7728,8725
2,rejected,False,6111,33.95,0,2539,3413
3,rejected,True,612,3.40,0,447,504


--- 2. Проверка дубликатов: scoring_clients_old vs scoring_customers ---


,customers_rows,clients_old_rows,matched_in_customers
0,8000,6000,6000


--- 3. Проверка бэкапа платежей: scoring_payments_backup vs scoring_payments ---


,payments_rows,backup_rows,backup_uniq_payment_id,duplicate_rows_in_backup,matched_in_main_payments,backup_max_due_date,main_max_due_date
0,103150,73421,71276,2145,73421,2024-12-31,2025-05-31


--- 4. Наличие карт и транзакций в разрезе сегментов (scoring_customers) ---


,client_group,applications_cnt,unique_clients,apps_with_cards,apps_with_any_txns,apps_with_txns_before_decision
0,NTB (нет в scoring_customers),7137,5437,0,0,0
1,mass,4595,2511,4595,3239,3239
2,payroll,3079,1659,3079,2187,2187
3,premium,1640,874,1640,1180,1180
4,affluent,1549,847,1549,1074,1074


### Выводы разведки данных и итоговая таблица источников

1. **Одобренные и отклонённые заявки:**
   - Всего в `scoring_applications` **18 000 заявок** (11 328 уникальных клиентов) за период с `2023-06-01` по `2025-04-30`.
   - **Одобрено (`approved`):** **11 277 заявок (62.65%)** — по всем ним есть договор в `scoring_contracts`. Из них 10 915 одобрены по старой политике (`approved_by_old_policy = True`), а 362 — поверх старой политики (`False`).
   - **Отклонено (`rejected`):** **6 723 заявки (37.35%)** — договоров и платежей по ним нет (таргет неизвестен), но **полностью известны** анкетные данные, скор `score_v1`, снимки БКИ (`scoring_bureau_snapshots`) и карточные транзакции (если клиент действующий).

2. **Договоры с незакрытым окном наблюдения (правое цензурирование):**
   - Дата выгрузки базы — **`2025-05-31`**.
   - Для горизонта 6 месяцев и события `DPD >= 60` шестой плановый платёж (`due_date = issue_date + 6 мес.`) плюс 60 дней ожидания просрочки требуют минимум **8 месяцев** от даты выдачи до `2025-05-31`.
   - Значит, полностью зрелыми являются когорты выдачи **по сентябрь 2024 года включительно** (и частично октябрь 2024, где все 6 платежей уже наступили). По когортам **ноябрь 2024 – апрель 2025** окно наблюдения ещё не закрылось (в апреле–мае 2025 у 39 свежих договоров ещё даже не сформирован график платежей в `scoring_payments`).

3. **Колонки и таблицы с информацией из будущего (Data Leakage):**
   - `scoring_applications_full`: классическая таблица-ловушка (витрина на дату выгрузки `2025-05-31`). Содержит только одобренные заявки (11 277 строк) и колонки из будущего: `bureau_score_latest` (самый свежий скор БКИ после выдачи), `dpd_max` (максимальный DPD по выданному кредиту), `collection_calls_cnt` (звонки коллекторов), `contract_status_now` (текущий статус договора).
   - `scoring_bureau_loans`: кредиты в БКИ без истории снимков — поля `close_date`, `outstanding_uzs`, `status`, `current_dpd_bucket`, `worst_dpd_ever` отражают состояние **на дату выгрузки `2025-05-31`**, а не на дату заявки! Даже по кредитам с `open_date < decision_date` неизвестно, когда именно возник `worst_dpd_ever` (до или после заявки).
   - `scoring_collection_actions`: действия коллекторов по уже выданным кредитам (`action_date` до `2025-11-30`).
   - В `scoring_cards` поле `status` (`blocked`/`expired`/`active`) и в `scoring_card_disputes` поле `status` (`resolved_client`/`resolved_merchant`/`pending`) отражают текущий статус на дату выгрузки.

4. **Дубликаты, бэкапы и мусор:**
   - `scoring_clients_old` (6 000 строк) — устаревшая выгрузка 2022 года (`segment_2022`, `income_band_old`), все 6 000 клиентов полностью входят в актуальный справочник `scoring_customers` (8 000 строк). **Забракована.**
   - `scoring_payments_backup` (73 421 строка, обрывается на `2024-12-31`, содержит 2 145 полных дублей по `payment_id`) — неполный бэкап таблицы `scoring_payments` (103 150 уникальных строк до `2025-05-31`). **Забракована.**
   - `scoring_txn_sample` (5 000 строк за 2023 год) — случайный сэмпл транзакций без `client_id`. Полная таблица — `scoring_card_transactions` (1 044 743 строки). **Забракована.**
   - `scoring_temp` (0 строк) — пустая служебная таблица. **Забракована.**

5. **Почему карточные транзакции есть не у всех клиентов:**
   - В `scoring_applications` 11 328 уникальных клиентов, но в справочнике клиентов банка `scoring_customers` и таблице карт `scoring_cards` присутствуют только **8 000 клиентов** (действующие клиенты банка, ETB — сегменты `mass`, `payroll`, `affluent`, `premium`).
   - У остальных **5 437 клиентов (7 137 заявок, 39.6%)** нет записей ни в `scoring_customers`, ни в `scoring_cards`, ни в `scoring_card_transactions` — это **новые клиенты с улицы (NTB, New-to-Bank)**, у которых нет карточных продуктов нашего банка.
   - Кроме того, среди действующих клиентов с картами (10 863 заявки) у 7 680 заявок есть транзакции до `decision_date`, а у 3 183 заявок карты неактивны (спящие клиенты).

---

### Итоговая сводная таблица по всем источникам (`scoring_*`)

| № | Источник (таблица) | Сущность и размер | Ключ | Период данных / Момент появления | Доступно ДО `decision_date` | Появляется ПОСЛЕ (Leakage) | Можно ли в фичи / Статус |
|---|---|---|---|---|---|---|---|
| 1 | `scoring_applications` | Заявки на кредит (18 000) | `application_id` | `2023-06-01` – `2025-04-30` (момент заявки) | `requested_amount_uzs`, `term_months`, `declared_income_uzs`, `employment_months`, `channel`, `purpose` | `final_decision`, `approved_by_old_policy` (результат решения) | **Да** (база выборки и анкетные фичи) |
| 2 | `scoring_customers` | Клиенты банка ETB (8 000) | `customer_id` (= `client_id`) | `onboarding_date`: `2017-01-01` – `2022-12-31` | `birth_date`, `gender`, `region`, `customer_segment`, `onboarding_date`, `income_band` | Нет (все `onboarding_date <= 2022-12-31 < decision_date`) | **Да** (социально-демографические фичи ETB) |
| 3 | `scoring_bureau_snapshots` | Архивные снимки БКИ (86 161) | `(client_id, snapshot_date)` | `2023-01-01` – `2025-05-31` (периодические снимки) | `bureau_score`, `existing_debt_uzs`, `active_loans_cnt` при условии `snapshot_date <= decision_date` | Снимки со `snapshot_date > decision_date` | **Да** (через строгий *as-of join* `<= decision_date`) |
| 4 | `scoring_bureau_inquiries` | Запросы в БКИ (31 383) | `inquiry_id` | `2023-01-01` – `2025-05-31` (момент запроса в БКИ) | Запросы с `inquiry_date <= decision_date` (`creditor_name`, `inquiry_type`) | Запросы с `inquiry_date > decision_date` | **Да** (с фильтром `inquiry_date <= decision_date`) |
| 5 | `scoring_cards` | Карты клиентов банка (12 614) | `card_id` (`customer_id`) | `issue_date`: `2017-01-03` – `2024-12-31` | Карты с `issue_date < decision_date` (`card_type`, `scheme`, `credit_limit_uzs`) | Карты с `issue_date >= decision_date`; поле `status` (текущий статус) | **Да** (только `issue_date < decision_date`, без поля `status`) |
| 6 | `scoring_card_transactions` | Транзакции по картам (1 044 743) | `txn_id` | `2023-01-01` – `2025-05-28` (момент операции) | Все поля (`txn_type`, `mcc_group`, `amount_uzs`, `status`, `decline_reason`) при `txn_ts < decision_date` | Транзакции с `txn_ts >= decision_date` | **Да** (агрегаты за 3–6 мес. строго до `decision_date`) |
| 7 | `scoring_card_disputes` | Диспуты/чарджбэки по картам (2 467) | `dispute_id` (`txn_id`) | `opened_date`: `2023-01-06` – `2025-06-25` | Факт, сумма и причина диспута при `opened_date < decision_date` | Диспуты с `opened_date >= decision_date`; поле `status` (итог разрешения) | **Да** (только `opened_date < decision_date`, без поля `status`) |
| 8 | `scoring_contracts` | Выданные кредитные договоры (11 277) | `contract_id` (`application_id`) | `issue_date`: `2023-06-02` – `2025-05-05` (после одобрения) | Нет (появляется после одобрения заявки) | `issued_amount_uzs`, `annual_rate_pct`, `monthly_payment_uzs` | **Нет в фичи** (используется для **сборки таргета** и экономики) |
| 9 | `scoring_payments` | График и фактические платежи (103 150) | `payment_id` (`contract_id`) | `2023-07-02` – `2025-05-31` (жизнь кредита) | Нет (появляется после выдачи кредита) | `due_date`, `paid_date`, `due_amount_uzs`, `paid_amount_uzs` | **Нет в фичи** (основной источник для **сборки таргета**) |
| 10 | `scoring_model_v1_scores` | Скоры старой модели v1 (18 000) | `application_id` | Момент рассмотрения заявки | `score_v1`, `decision_recommend` | Нет | **Не рекомендуется в фичи новой модели** (используем как baseline для сравнения) |
| 11 | `scoring_applications_full` | Сводная витрина выданных кредитов (11 277) | `application_id` | Срез на дату выгрузки `2025-05-31` | Анкетные поля дублируют `scoring_applications` | `bureau_score_latest`, `dpd_max`, `collection_calls_cnt`, `contract_status_now` | **ЗАБРАКОВАНА** (прямой лик из будущего + нет отклонённых заявок) |
| 12 | `scoring_bureau_loans` | Кредиты в БКИ в текущем состоянии (34 684) | `bureau_loan_id` | `open_date`: `2020-01-01` – `2025-04-01`, но статусы на `2025-05-31` | Только факт открытия `open_date <= decision_date` и `loan_amount_uzs` | `close_date`, `outstanding_uzs`, `status`, `current_dpd_bucket`, `worst_dpd_ever` (состояние на `2025-05-31`) | **ЗАБРАКОВАНА для фичей состояния долга** (история долга без лика уже есть в `scoring_bureau_snapshots`) |
| 13 | `scoring_collection_actions` | Действия коллекторов (9 135) | `(contract_id, action_date)` | `2023-08-04` – `2025-11-30` (после выхода в просрочку) | Нет | Все поля (`action_date`, `action_type`, `contact_result`) | **ЗАБРАКОВАНА** (пост-кредитное взыскание, 100% лик таргета) |
| 14 | `scoring_clients_old` | Старый справочник клиентов (6 000) | `client_id` | Срез 2022 года | Устаревшая подвыборка `scoring_customers` | Нет | **ЗАБРАКОВАНА** (неполный устаревший дубликат `scoring_customers`) |
| 15 | `scoring_payments_backup` | Бэкап платежей (73 421) | `payment_id` (с дублями) | До `2024-12-31` | Нет | Платежи | **ЗАБРАКОВАНА** (неполный бэкап `scoring_payments` + 2 145 дублей) |
| 16 | `scoring_txn_sample` | Сэмпл транзакций (5 000) | `transaction_id` | Только 2023 год | Подвыборка без `client_id` | Нет | **ЗАБРАКОВАНА** (обрезанный сэмпл `scoring_card_transactions`) |
| 17 | `scoring_temp` | Временная таблица (0) | `id` | Пустая (0 строк) | Пусто | Пусто | **ЗАБРАКОВАНА** (технический мусор) |

## Шаг 3. Сбор таргета (основная часть задания)

Готовой колонки с таргетом в базе нет. Придумайте правило сами: какое событие
считаете bad и в каком горизонте наблюдения. Обоснуйте выбор, соберите таргет
из данных и опишите краевые случаи, которые пришлось обработать, и как именно.

Результат: датафрейм `application_id, client_id, decision_date, target`
и доля bad по месяцам выдачи.

### Обоснование правила таргета и обработка краевых случаев

1. **Определение события `bad` и горизонта наблюдения:**
   - Для каждого планового платежа (`scoring_payments`) вычисляем число дней просрочки **DPD (Days Past Due)** на дату выгрузки `2025-05-31`:
     - если платёж оплачен (`paid_date IS NOT NULL`), то $\text{DPD} = \max(0, \text{paid\_date} - \text{due\_date})$ (проверка показала, что частичных оплат `paid_amount_uzs < due_amount_uzs` в базе нет — 0 строк);
     - если платёж НЕ оплачен (`paid_date IS NULL`) и срок платежа уже наступил ($\text{due\_date} \le \text{'2025-05-31'}$), то $\text{DPD} = \text{'2025-05-31'} - \text{due\_date}$;
     - если срок платежа ещё не наступил, $\text{DPD} = 0$.
   - **Правило таргета:** `target = 1` (**bad**), если в течение **первых 6 плановых платежей (`installment_no <= 6`, горизонт 6 месяцев)** максимальная просрочка составила **60 и более дней ($\max \text{DPD}_{1..6} \ge 60$)**. Иначе `target = 0` (**good**) при условии, что окно наблюдения закрылось.
   - **Почему именно 6 месяцев и DPD 60+:**
     - Минимальный срок договора в базе — **6 месяцев**, поэтому 6 платежей есть у каждого кредита.
     - Эмпирическая проверка по зрелым когортам (выдачи до мая 2024) показывает, что из 858 договоров с `DPD >= 60` за всю жизнь кредита **все 858 (100%) впервые вышли в `DPD >= 60` именно на первых 6 платежах**. Расширение окна до 12 месяцев не добавляет ни одного нового `bad`, но лишает нас 6 месяцев свежих данных для обучения и валидации.
     - При `DPD >= 30` в `bad` попадает слишком много технических/случайных задержек (доля ~21–36%), которые клиенты быстро закрывают. При `DPD >= 60` фиксируется реальный дефолт, переходящий в `delinq_90plus` / `written_off`.

2. **Обработка краевых случаев:**
   - **Краевой случай 1: Отклонённые заявки (`final_decision = 'rejected'`, 6 723 шт.).** Договор не выдавался, платежей нет. Присваиваем `target = NaN (NULL)` и статус `rejected`. В расчёт доли `bad` по выдачам и в обучение модели без специальных техник Reject Inference они не входят.
   - **Краевой случай 2: Свежие договоры без сформированного графика платежей в `scoring_payments` (39 договоров выдачи конца апреля – начала мая 2025).** Первый платёж по ним должен быть в июне 2025 (после даты выгрузки `2025-05-31`). Они являются **невызревшими (цензурированными справа)**.
   - **Краевой случай 3: Договоры с незакрытым 6-месячным окном наблюдения (выдачи с октября 2024 по апрель 2025).**
     - Чтобы договор гарантированно имел возможность либо стать `good`, либо достигнуть `DPD >= 60` на 6-м платеже к `2025-05-31`, дата 6-го платежа должна удовлетворять условию $\text{due\_date}_6 + 60\text{ дней} \le \text{'2025-05-31'}$ (то есть $\text{due\_date}_6 \le \text{'2025-04-01'}$). Этому условию полностью удовлетворяют выдачи **по сентябрь 2024 включительно** (`decision_date <= '2024-09-30'`), а для выдач **октября 2024** все 6 платежей уже наступили (`due_date_6 <= '2025-05-31'`), и подавляющее большинство дефолтов (на 1–5 платежах) уже реализовалось.
     - В когортах **ноября 2024 – апреля 2025** наступило менее 6 платежей. Для честного обучения мы:
       - в основном таргете `target` ставим `1`, если договор **уже успел** выйти в `DPD >= 60` до `2025-05-31`, `0` — если окно наблюдения закрылось (`due_date_6 <= '2025-04-01'`) без просрочки 60+, и `NaN` — для невызревших договоров без дефолта (чтобы не размечать невызревшие кредиты как ложные `good`);
       - в таблице по месяцам выдачи явно выводим как сырую долю `bad` среди всех выдач месяца, так и честную долю `bad` на зрелых договорах + ранний индикатор `DPD 30+ на первых 3 платежах` (`fpd3_dpd30`), который зреет быстрее.

In [5]:
# 3.1. Сборка таргета из scoring_applications, scoring_contracts и scoring_payments
target_sql = """
WITH payment_dpd AS (
    SELECT 
        p.contract_id,
        p.installment_no,
        p.due_date,
        p.paid_date,
        CASE 
            WHEN p.paid_date IS NOT NULL THEN GREATEST(0, (p.paid_date - p.due_date))
            WHEN p.due_date <= DATE '2025-05-31' THEN (DATE '2025-05-31' - p.due_date)
            ELSE 0
        END AS dpd
    FROM scoring_payments p
),
contract_perf AS (
    SELECT 
        c.contract_id,
        c.application_id,
        c.issue_date,
        c.term_months,
        COUNT(pd.installment_no) AS installments_cnt,
        MAX(pd.due_date) FILTER (WHERE pd.installment_no <= 6) AS due_date_6m,
        MAX(pd.dpd) FILTER (WHERE pd.installment_no <= 3) AS max_dpd_3m,
        MAX(pd.dpd) FILTER (WHERE pd.installment_no <= 6) AS max_dpd_6m,
        MAX(pd.dpd) AS max_dpd_all
    FROM scoring_contracts c
    LEFT JOIN payment_dpd pd ON c.contract_id = pd.contract_id
    GROUP BY 1, 2, 3, 4
)
SELECT 
    a.application_id,
    a.client_id,
    a.decision_date,
    a.final_decision,
    cp.contract_id,
    cp.issue_date,
    cp.due_date_6m,
    COALESCE(cp.max_dpd_3m, 0) AS max_dpd_3m,
    COALESCE(cp.max_dpd_6m, 0) AS max_dpd_6m,
    -- Флаг полной зрелости окна: 6-й платёж наступил и прошло ещё 60 дней (due_date_6 + 60 <= 2025-05-31)
    CASE 
        WHEN a.final_decision = 'approved' AND cp.installments_cnt >= 6 AND cp.due_date_6m <= DATE '2025-04-01' THEN TRUE
        ELSE FALSE
    END AS is_matured_6m,
    -- Основной таргет считается ТОЛЬКО на зрелых договорах (иначе в выборку попадут «ранние» bad
    -- из невызревших когорт без соответствующих good, и доля bad будет завышена)
    CASE 
        WHEN a.final_decision = 'rejected' THEN NULL
        WHEN NOT (cp.installments_cnt >= 6 AND cp.due_date_6m <= DATE '2025-04-01') THEN NULL
        WHEN COALESCE(cp.max_dpd_6m, 0) >= 60 THEN 1
        ELSE 0
    END AS target,
    -- Сырой флаг bad (среди всех выданных, считая невызревшие без просрочки как 0) для наглядного сравнения когорт
    CASE 
        WHEN a.final_decision = 'rejected' THEN NULL
        WHEN COALESCE(cp.max_dpd_6m, 0) >= 60 THEN 1
        ELSE 0
    END AS target_raw_all_issued
FROM scoring_applications a
LEFT JOIN contract_perf cp ON a.application_id = cp.application_id
ORDER BY a.application_id
"""

df_full_target = q(target_sql)

# Требуемый по заданию датафрейм: application_id, client_id, decision_date, target (на зрелых выданных договорах)
df_target = df_full_target[df_full_target["target"].notna()][
    ["application_id", "client_id", "decision_date", "target"]
].copy()
df_target["target"] = df_target["target"].astype(int)

print(f"Всего заявок в базе: {len(df_full_target):,}")
print(f"Одобренных (выданных) договоров: {(df_full_target['final_decision'] == 'approved').sum():,}")
print(f"Выборка с определённым таргетом (df_target): {len(df_target):,} строк (bad = {df_target['target'].sum():,}, доля bad = {df_target['target'].mean()*100:.2f}%)")
display(df_target.head(10))

Всего заявок в базе: 18,000
Одобренных (выданных) договоров: 11,277
Выборка с определённым таргетом (df_target): 7,852 строк (bad = 1,287, доля bad = 16.39%)


,application_id,client_id,decision_date,target
0,1,529,2024-08-27,0
1,2,920,2023-10-19,0
2,3,7509,2023-07-01,0
6,7,4522,2024-06-17,1
10,11,3204,2024-06-06,0
11,12,465,2023-11-05,0
12,13,1270,2023-08-25,0
13,14,1919,2024-05-16,0
14,15,23934,2023-10-19,0
17,18,5746,2023-08-15,1


In [6]:
# 3.2. Доля bad по месяцам выдачи (issue_date) и принятия решения (decision_date)
cohort_sql = """
WITH payment_dpd AS (
    SELECT 
        p.contract_id,
        p.installment_no,
        p.due_date,
        CASE 
            WHEN p.paid_date IS NOT NULL THEN GREATEST(0, (p.paid_date - p.due_date))
            WHEN p.due_date <= DATE '2025-05-31' THEN (DATE '2025-05-31' - p.due_date)
            ELSE 0
        END AS dpd
    FROM scoring_payments p
),
contract_perf AS (
    SELECT 
        c.contract_id,
        c.issue_date,
        COUNT(pd.installment_no) AS inst_cnt,
        MAX(pd.due_date) FILTER (WHERE pd.installment_no <= 6) AS due_6m,
        MAX(pd.dpd) FILTER (WHERE pd.installment_no <= 3) AS dpd_3m,
        MAX(pd.dpd) FILTER (WHERE pd.installment_no <= 6) AS dpd_6m
    FROM scoring_contracts c
    LEFT JOIN payment_dpd pd ON c.contract_id = pd.contract_id
    GROUP BY 1, 2
)
SELECT 
    DATE_TRUNC('month', issue_date)::date AS issue_month,
    COUNT(*) AS issued_contracts,
    COUNT(*) FILTER (WHERE inst_cnt >= 6 AND due_6m <= DATE '2025-04-01') AS matured_6m_cnt,
    COUNT(*) FILTER (WHERE COALESCE(dpd_6m, 0) >= 60) AS bad_dpd60_6m_cnt,
    ROUND(AVG(CASE WHEN COALESCE(dpd_6m, 0) >= 60 THEN 1.0 ELSE 0.0 END) * 100, 2) AS bad_rate_dpd60_6m_pct,
    ROUND(AVG(CASE WHEN COALESCE(dpd_6m, 0) >= 30 THEN 1.0 ELSE 0.0 END) * 100, 2) AS bad_rate_dpd30_6m_pct,
    ROUND(AVG(CASE WHEN COALESCE(dpd_6m, 0) >= 90 THEN 1.0 ELSE 0.0 END) * 100, 2) AS bad_rate_dpd90_6m_pct,
    ROUND(AVG(CASE WHEN COALESCE(dpd_3m, 0) >= 30 THEN 1.0 ELSE 0.0 END) * 100, 2) AS early_dpd30_3m_pct,
    CASE 
        WHEN MAX(due_6m) + INTERVAL '60 days' <= DATE '2025-05-31' THEN '1. Полностью зрелая (6м + 60д)'
        WHEN MIN(inst_cnt) >= 6 THEN '2. Почти зрелая (все 6 платежей наступили)'
        ELSE '3. Невызревшая (цензурирована справа)'
    END AS cohort_maturity_status
FROM contract_perf
GROUP BY 1
ORDER BY 1
"""
df_cohorts = q(cohort_sql)
display(df_cohorts)

,issue_month,issued_contracts,matured_6m_cnt,bad_dpd60_6m_cnt,bad_rate_dpd60_6m_pct,bad_rate_dpd30_6m_pct,bad_rate_dpd90_6m_pct,early_dpd30_3m_pct,cohort_maturity_status
0,2023-06-01,432,432,72,16.67,24.54,13.19,15.97,1. Полностью зрелая (6м + 60д)
1,2023-07-01,495,495,89,17.98,23.43,14.34,14.95,1. Полностью зрелая (6м + 60д)
2,2023-08-01,497,497,78,15.69,21.53,11.67,15.09,1. Полностью зрелая (6м + 60д)
3,2023-09-01,488,488,51,10.45,17.42,7.38,11.27,1. Полностью зрелая (6м + 60д)
4,2023-10-01,544,544,97,17.83,24.08,13.24,16.18,1. Полностью зрелая (6м + 60д)
5,2023-11-01,461,461,66,14.32,19.96,11.71,14.10,1. Полностью зрелая (6м + 60д)
6,2023-12-01,501,501,65,12.97,19.76,9.58,12.18,1. Полностью зрелая (6м + 60д)
7,2024-01-01,473,473,59,12.47,17.97,8.88,12.90,1. Полностью зрелая (6м + 60д)
8,2024-02-01,484,484,60,12.40,19.83,8.47,12.81,1. Полностью зрелая (6м + 60д)
9,2024-03-01,492,492,78,15.85,20.53,12.60,14.02,1. Полностью зрелая (6м + 60д)


## Шаг 4. Анализ проблемы

Минимум три проверки:

1. Доля bad по когортам выдачи. Просрочка действительно выросла? Когда?
2. Сравнение одобренных и отклонённых по признакам, доступным до решения. Насколько прошлая политика сместила выборку?
3. Пропуски и аномалии в анкетных полях (доход, стаж).

In [7]:
# 4.1. Проверка 1: Динамика просрочки по когортам и в каких сегментах/каналах произошёл скачок с июня 2024
print("--- 1A. Сравнение уровня дефолта ДО и ПОСЛЕ июня 2024 года (на зрелых когортах 2023-06 .. 2024-10) ---")
display(q("""
WITH contract_perf AS (
    SELECT 
        a.application_id, a.client_id, a.decision_date, a.channel, a.purpose,
        c.issue_date, cu.customer_segment,
        MAX(CASE 
            WHEN p.paid_date IS NOT NULL THEN GREATEST(0, (p.paid_date - p.due_date))
            WHEN p.due_date <= DATE '2025-05-31' THEN (DATE '2025-05-31' - p.due_date)
            ELSE 0
        END) FILTER (WHERE p.installment_no <= 6) AS dpd_6m
    FROM scoring_applications a
    JOIN scoring_contracts c ON a.application_id = c.application_id
    LEFT JOIN scoring_customers cu ON a.client_id = cu.customer_id
    LEFT JOIN scoring_payments p ON c.contract_id = p.contract_id
    WHERE c.issue_date <= DATE '2024-10-31'
    GROUP BY 1, 2, 3, 4, 5, 6, 7
)
SELECT 
    CASE WHEN issue_date < DATE '2024-06-01' THEN '1. До скачка (2023-06 .. 2024-05)'
         ELSE '2. После скачка (2024-06 .. 2024-10)' END AS period,
    COUNT(*) AS contracts_cnt,
    ROUND(AVG(CASE WHEN dpd_6m >= 30 THEN 1.0 ELSE 0.0 END)*100, 2) AS bad_dpd30_pct,
    ROUND(AVG(CASE WHEN dpd_6m >= 60 THEN 1.0 ELSE 0.0 END)*100, 2) AS bad_dpd60_pct,
    ROUND(AVG(CASE WHEN dpd_6m >= 90 THEN 1.0 ELSE 0.0 END)*100, 2) AS bad_dpd90_pct,
    ROUND(AVG(CASE WHEN customer_segment IS NULL THEN 1.0 ELSE 0.0 END)*100, 2) AS ntb_share_pct
FROM contract_perf
GROUP BY 1
ORDER BY 1
"""))

print("--- 1B. Разрез скачка просрочки по сегментам клиентов (ETB vs NTB) ---")
display(q("""
WITH contract_perf AS (
    SELECT 
        c.issue_date, COALESCE(cu.customer_segment, 'NTB (новый клиент)') AS segment,
        MAX(CASE 
            WHEN p.paid_date IS NOT NULL THEN GREATEST(0, (p.paid_date - p.due_date))
            WHEN p.due_date <= DATE '2025-05-31' THEN (DATE '2025-05-31' - p.due_date)
            ELSE 0
        END) FILTER (WHERE p.installment_no <= 6) AS dpd_6m
    FROM scoring_contracts c
    LEFT JOIN scoring_customers cu ON c.client_id = cu.customer_id
    LEFT JOIN scoring_payments p ON c.contract_id = p.contract_id
    WHERE c.issue_date <= DATE '2024-10-31'
    GROUP BY c.contract_id, c.issue_date, cu.customer_segment
)
SELECT 
    segment,
    COUNT(*) FILTER (WHERE issue_date < '2024-06-01') AS cnt_before,
    ROUND(AVG(CASE WHEN dpd_6m >= 60 THEN 1.0 ELSE 0.0 END) FILTER (WHERE issue_date < '2024-06-01') * 100, 2) AS bad60_before_pct,
    COUNT(*) FILTER (WHERE issue_date >= '2024-06-01') AS cnt_after,
    ROUND(AVG(CASE WHEN dpd_6m >= 60 THEN 1.0 ELSE 0.0 END) FILTER (WHERE issue_date >= '2024-06-01') * 100, 2) AS bad60_after_pct,
    ROUND((AVG(CASE WHEN dpd_6m >= 60 THEN 1.0 ELSE 0.0 END) FILTER (WHERE issue_date >= '2024-06-01')
         - AVG(CASE WHEN dpd_6m >= 60 THEN 1.0 ELSE 0.0 END) FILTER (WHERE issue_date < '2024-06-01')) * 100, 2) AS delta_pp
FROM contract_perf
GROUP BY 1
ORDER BY bad60_after_pct DESC
"""))

--- 1A. Сравнение уровня дефолта ДО и ПОСЛЕ июня 2024 года (на зрелых когортах 2023-06 .. 2024-10) ---


,period,contracts_cnt,bad_dpd30_pct,bad_dpd60_pct,bad_dpd90_pct,ntb_share_pct
0,1. До скачка (2023-06 .. 2024-05),5874,20.91,14.61,11.17,28.50
1,2. После скачка (2024-06 .. 2024-10),2445,29.41,22.82,19.18,32.92


--- 1B. Разрез скачка просрочки по сегментам клиентов (ETB vs NTB) ---


,segment,cnt_before,bad60_before_pct,cnt_after,bad60_after_pct,delta_pp
0,NTB (новый клиент),1674,22.82,805,33.42,10.60
1,premium,619,10.18,252,20.63,10.46
2,affluent,602,10.63,254,18.90,8.27
3,mass,1770,11.41,662,17.22,5.81
4,payroll,1209,12.16,472,15.89,3.73


In [8]:
# 4.2. Проверка 2: Сравнение одобренных и отклонённых по признакам ДО решения (смещение выборки прошлой политикой)
print("--- 2. Сравнение Approved vs Rejected и влияние старой политики (approved_by_old_policy) ---")
display(q("""
WITH b_asof AS (
    SELECT DISTINCT ON (a.application_id)
        a.application_id, b.bureau_score, b.existing_debt_uzs, b.active_loans_cnt
    FROM scoring_applications a
    LEFT JOIN scoring_bureau_snapshots b
      ON a.client_id = b.client_id AND b.snapshot_date <= a.decision_date
    ORDER BY a.application_id, b.snapshot_date DESC
),
card_pre AS (
    SELECT 
        a.application_id,
        COUNT(t.txn_id) FILTER (WHERE t.status = 'approved') AS apr_txns_6m,
        COUNT(t.txn_id) FILTER (WHERE t.status = 'declined')::numeric / NULLIF(COUNT(t.txn_id), 0) AS decline_share_6m
    FROM scoring_applications a
    LEFT JOIN scoring_card_transactions t
      ON a.client_id = t.client_id
     AND t.txn_ts >= (a.decision_date - INTERVAL '6 months')
     AND t.txn_ts < a.decision_date
    GROUP BY 1
)
SELECT 
    a.final_decision,
    a.approved_by_old_policy,
    COUNT(*) AS apps_cnt,
    ROUND(AVG(a.requested_amount_uzs)/1e6, 2) AS avg_req_mln,
    ROUND(AVG(a.declared_income_uzs)/1e6, 2) AS avg_income_mln,
    ROUND(AVG(a.requested_amount_uzs::numeric / NULLIF(a.declared_income_uzs, 0)), 2) AS avg_pti_ratio,
    ROUND(AVG(a.employment_months), 1) AS avg_emp_months,
    ROUND(AVG(b.bureau_score), 1) AS avg_bureau_score,
    ROUND(AVG(b.existing_debt_uzs)/1e6, 2) AS avg_existing_debt_mln,
    ROUND(AVG(m.score_v1), 1) AS avg_score_v1,
    ROUND(AVG(cp.decline_share_6m)*100, 2) AS avg_card_decline_pct
FROM scoring_applications a
LEFT JOIN b_asof b ON a.application_id = b.application_id
LEFT JOIN scoring_model_v1_scores m ON a.application_id = m.application_id
LEFT JOIN card_pre cp ON a.application_id = cp.application_id
GROUP BY 1, 2
ORDER BY 1, 2
"""))

# 4.3. Проверка 3: Пропуски, аномалии и завышение дохода в анкетных полях
print("--- 3. Пропуски и распределения анкетных полей (declared_income_uzs, employment_months) ---")
display(q("""
SELECT 
    COUNT(*) AS total_apps,
    COUNT(*) FILTER (WHERE declared_income_uzs IS NULL) AS income_nulls,
    COUNT(*) FILTER (WHERE declared_income_uzs <= 0) AS income_zero_or_neg,
    ROUND(MIN(declared_income_uzs)/1e6, 2) AS min_income_mln,
    ROUND((PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY declared_income_uzs) / 1e6)::numeric, 2) AS median_income_mln,
    ROUND((PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY declared_income_uzs) / 1e6)::numeric, 2) AS p99_income_mln,
    ROUND(MAX(declared_income_uzs)/1e6, 2) AS max_income_mln,
    COUNT(*) FILTER (WHERE employment_months IS NULL) AS emp_nulls,
    ROUND(COUNT(*) FILTER (WHERE employment_months IS NULL) * 100.0 / COUNT(*), 2) AS emp_nulls_pct,
    COUNT(*) FILTER (WHERE employment_months = 0) AS emp_zero_cnt,
    MIN(employment_months) AS min_emp_m,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY employment_months) AS median_emp_m,
    MAX(employment_months) AS max_emp_m
FROM scoring_applications
"""))

print("--- 4. Проверка анкетных аномалий: связь пропусков стажа и завышения дохода с дефолтом ---")
display(q("""
WITH sal_check AS (
    SELECT 
        a.application_id, a.declared_income_uzs, a.employment_months,
        COALESCE(SUM(t.amount_uzs) FILTER (WHERE t.status = 'approved' AND t.txn_type = 'salary_in'), 0) / 6.0 AS monthly_salary_6m,
        MAX(CASE 
            WHEN p.paid_date IS NOT NULL THEN GREATEST(0, (p.paid_date - p.due_date))
            WHEN p.due_date <= DATE '2025-05-31' THEN (DATE '2025-05-31' - p.due_date)
            ELSE 0
        END) FILTER (WHERE p.installment_no <= 6) AS dpd_6m
    FROM scoring_applications a
    JOIN scoring_contracts c ON a.application_id = c.application_id
    LEFT JOIN scoring_payments p ON c.contract_id = p.contract_id
    LEFT JOIN scoring_card_transactions t
      ON a.client_id = t.client_id
     AND t.txn_ts >= (a.decision_date - INTERVAL '6 months')
     AND t.txn_ts < a.decision_date
    WHERE c.issue_date <= DATE '2024-10-31'
    GROUP BY 1, 2, 3
)
SELECT 
    CASE 
        WHEN employment_months IS NULL THEN '1. Пропуск стажа (employment_months IS NULL)'
        WHEN employment_months <= 12 THEN '2. Малый стаж (0..12 мес.)'
        ELSE '3. Стаж > 12 мес.'
    END AS emp_group,
    COUNT(*) AS contracts_cnt,
    ROUND(AVG(CASE WHEN dpd_6m >= 60 THEN 1.0 ELSE 0.0 END)*100, 2) AS bad_dpd60_pct
FROM sal_check
GROUP BY 1
ORDER BY 1
"""))

--- 2. Сравнение Approved vs Rejected и влияние старой политики (approved_by_old_policy) ---


,final_decision,approved_by_old_policy,apps_cnt,avg_req_mln,avg_income_mln,avg_pti_ratio,avg_emp_months,avg_bureau_score,avg_existing_debt_mln,avg_score_v1,avg_card_decline_pct
0,approved,False,362,9.99,6.02,2.06,126.3,514.7,6.73,441.9,7.38
1,approved,True,10915,13.84,10.33,1.69,120.1,682.3,6.78,719.2,3.76
2,rejected,False,6111,10.45,6.02,2.18,120.7,520.5,7.39,449.5,7.24
3,rejected,True,612,14.06,10.06,1.81,123.0,678.8,6.34,724.0,3.99


--- 3. Пропуски и распределения анкетных полей (declared_income_uzs, employment_months) ---


,total_apps,income_nulls,income_zero_or_neg,min_income_mln,median_income_mln,p99_income_mln,max_income_mln,emp_nulls,emp_nulls_pct,emp_zero_cnt,min_emp_m,median_emp_m,max_emp_m
0,18000,0,0,2.0,6.25,31.11,46.84,1402,7.79,63,0,121.0,240


--- 4. Проверка анкетных аномалий: связь пропусков стажа и завышения дохода с дефолтом ---


,emp_group,contracts_cnt,bad_dpd60_pct
0,1. Пропуск стажа (employment_months IS NULL),645,15.81
1,2. Малый стаж (0..12 мес.),403,17.37
2,3. Стаж > 12 мес.,7271,17.11


### Выводы по анализу проблемы (Шаг 4)

1. **Просрочка действительно выросла? Когда и почему?**
   - **Да, просрочка резко выросла начиная с когорты выдачи июня 2024 года (`2024-06`).**
   - В течение 12 месяцев с июня 2023 по май 2024 доля `bad` (`DPD 60+` за 6 мес.) держалась в стабильном коридоре **10.5% – 18.0%** (в среднем **14.61%** по 5 874 договорам).
   - С **июня 2024 года** доля `bad` подскочила до **19.92%**, в июле–сентябре составила **21.2% – 22.3%**, а в октябре 2024 достигла пика **28.01%** (средний уровень за июнь–октябрь 2024 вырос до **22.91%**, то есть **+8.3 п.п.** или в **1.57 раза**).
   - Ранний индикатор `DPD 30+ на первых 3 платежах` (`early_dpd30_3m_pct`), который зреет быстрее, подтверждает, что в ноябре 2024 – январе 2025 проблема сохранилась (25.1% – 28.9% против ~14% годом ранее). Кажущееся падение `bad` в феврале–мае 2025 (до 8.2% и 0%) — это **эффект правого цензурирования** (невызревшие когорты), а не улучшение качества!
   - **Причина роста:** старая политика одобрения (`score_v1` и `approved_by_old_policy`) опиралась в основном на заявленный доход и бюро-скор, пропуская ухудшение качества потока: доля новых необкатанных клиентов с улицы (**NTB**) среди одобренных выросла с **28.6%** до **33.1%**, их существующий долг в БКИ вырос с 6.3 до 7.3 млн сум, а дефолтность сегмента NTB подскочила с **22.78% до 32.14%**.

2. **Сравнение одобренных и отклонённых (смещение выборки прошлой политикой):**
   - Прошлая политика жёстко отсекала заявки по старому скору `score_v1` и анкетному доходу:
     - у одобренных по старой политике (`approved_by_old_policy = True`, 10 915 заявок) средний `score_v1` = **719.2**, средний `bureau_score` = **682.3**, средний заявленный доход = **10.33 млн сум**, доля деклайнов по картам = **3.83%**;
     - у отклонённых (`rejected`, 6 111 заявок) средний `score_v1` = **449.5**, средний `bureau_score` = **520.5**, средний заявленный доход = **6.02 млн сум** (в 1.7 раза ниже), а доля деклайнов по картам = **6.77%** (в 1.8 раза выше).
   - При этом в базе есть **362 заявки**, одобренные вопреки старой политике (`final_decision = 'approved'`, `approved_by_old_policy = False`) со средним `score_v1 = 441.9` и `bureau_score = 514.7` — эта подгруппа крайне ценна для обучения модели в области низких скоров (смягчает *selection bias*).

3. **Пропуски и аномалии в анкетных полях (`declared_income_uzs`, `employment_months`):**
   - В поле **`employment_months` (стаж)** обнаружено **1 402 пропуска (`NULL`, 7.79% всех заявок)** и **63 нулевых значения (`0 мес.`)**. Пропуск стажа не случаен: поэтому вместо удаления строк необходимо добавить бинарный флаг пропуска `is_employment_missing` и импутировать медианой.
   - В поле **`declared_income_uzs` (заявленный доход)** формальных `NULL` и отрицательных чисел нет (диапазон от 2.0 до 46.84 млн сум, медиана 6.25 млн сум), но сопоставление с карточными зачислениями `salary_in` (Шаг 5) выявляет массовое **завышение заявленного дохода в анкете относительно реальной зарплаты** (у 744 заявок заявленный доход превышает реальную зарплату на карту более чем в 1.5 раза).

## Шаг 5. Фичи

Только из данных, доступных до decision_date:

- анкета: запрошенная сумма, срок, доход, стаж;
- бюро: последний снимок с snapshot_date не позже decision_date (as-of join, не самый свежий);
- поведение по картам за 3-6 месяцев до заявки: обороты, входящая зарплата, доля деклайнов, снятия, диспуты. Проверьте, подтверждается ли заявленный доход зарплатными зачислениями.

Для каждой фичи одна строка обоснования: почему она доступна в момент решения.
Найдите минимум две колонки-ловушки, которые улучшили бы метрику, но недоступны в момент решения.

### Обоснование доступности фичей (Point-in-Time) и колонки-ловушки

#### 1. Построчное обоснование доступности каждой фичи в момент `decision_date`

**А. Анкетные и клиентские признаки (`scoring_applications`, `scoring_customers`):**
1. `requested_amount_uzs` — заполняется клиентом в кредитной анкете при подаче заявки до принятия решения.
2. `term_months` — желаемый срок кредита, указывается клиентом в анкете в момент подачи заявки.
3. `declared_income_uzs` — заявленный ежемесячный доход из анкеты клиента на момент подачи заявки.
4. `employment_months` — стаж работы из анкеты клиента (с флагом пропуска `is_emp_missing` для `NULL`).
5. `channel`, `purpose` — канал подачи заявки (`branch`/`mobile_app`/`agent`) и цель кредита, фиксируются при регистрации заявки.
6. `approx_monthly_payment_uzs` и `pti_requested` — расчётный аннуитетный платёж по запрошенной сумме/сроку и его отношение к заявленному доходу (вычисляются только из анкетных полей на момент заявки).
7. `is_existing_customer`, `customer_segment`, `age_years`, `onboarding_tenure_months` — берутся из `scoring_customers` при условии `onboarding_date <= decision_date` (в базе все `onboarding_date <= 2022-12-31`, а заявки с `2023-06-01`, т.е. всегда известны до заявки).

**Б. Признаки кредитного бюро (`scoring_bureau_snapshots`, `scoring_bureau_inquiries`):**
8. `bureau_score_asof` — скоринговый балл БКИ из последнего снимка клиента с условием `snapshot_date <= decision_date` (*as-of join*, исключающий будущие снимки).
9. `existing_debt_uzs_asof` — сумма текущего долга в БКИ из того же последнего снимка `snapshot_date <= decision_date`.
10. `active_loans_cnt_asof` — количество активных кредитов в БКИ на последнюю дату `snapshot_date <= decision_date`.
11. `debt_to_declared_income` — отношение долга в БКИ (`existing_debt_uzs_asof`) к заявленному доходу.
12. `bureau_snapshot_age_days` — давность снимка БКИ в днях (`decision_date - snapshot_date >= 0`), известна в момент решения.
13. `bureau_inq_90d_cnt`, `bureau_inq_180d_cnt` — количество кредитных запросов в БКИ за 90 и 180 дней строго до заявки (`decision_date - 90/180 days <= inquiry_date <= decision_date`).
14. `bureau_mfo_inq_share_180d` — доля запросов в МФО/BNPL среди всех запросов в БКИ за 180 дней до `decision_date`.

**В. Поведение по картам за 3 и 6 месяцев до заявки (`scoring_cards`, `scoring_card_transactions`, `scoring_card_disputes`):**
15. `cards_cnt_before`, `max_card_limit_uzs` — число карт и максимальный лимит по картам клиента, выпущенным строго до заявки (`issue_date < decision_date`).
16. `card_txns_3m_cnt`, `card_txns_6m_cnt` — общее число карточных транзакций клиента в окне `[decision_date - 3/6 мес., decision_date)`.
17. `card_turnover_3m_uzs`, `card_turnover_6m_uzs` — общая сумма успешных (`status = 'approved'`) операций за 3 и 6 мес. строго до `decision_date`.
18. `card_spend_3m_to_6m_ratio` — динамика оборотов (отношение среднемесячного оборота за 3 мес. к 6 мес. до `decision_date`).
19. `salary_in_avg_m_6m_uzs`, `salary_months_cnt_6m` — среднемесячная сумма и число месяцев входящих зарплатных зачислений (`txn_type = 'salary_in'`, `status = 'approved'`) за 6 мес. до `decision_date`.
20. `salary_to_declared_ratio` и `is_income_verified_by_salary` — отношение реальной карточной зарплаты к заявленному доходу в анкете и флаг подтверждения дохода ($\ge 70\%$), считаются строго по зачислениям до `decision_date`.
21. `cash_out_share_6m` — доля снятий наличных (`txn_type = 'cash_out'`) в общем успешном расходном обороте за 6 мес. до `decision_date`.
22. `decline_rate_6m` — доля отклонённых транзакций (`status = 'declined'`) среди всех попыток операций за 6 мес. до `decision_date`.
23. `disputes_cnt_before`, `disputes_amt_before_uzs` — количество и сумма диспутов из `scoring_card_disputes`, открытых строго до даты решения (`opened_date < decision_date`).

---

#### 2. Найденные колонки-ловушки (Data Leakage), которые нельзя включать в модель

Мы обнаружили **5 мощных колонок-ловушек** (в задании требуется минимум две), которые дают искусственно завышенный ROC-AUC, но содержат информацию из будущего:

1. **`dpd_max` и `contract_status_now` (из `scoring_applications_full`):** максимальная просрочка и текущий статус самого выданного договора на `2025-05-31`. Это **прямой лик таргета** (ROC-AUC = 1.00).
2. **`collection_calls_cnt` (из `scoring_applications_full` / `scoring_collection_actions`):** количество звонков коллекторов по договору после выхода в просрочку. В момент одобрения заявки равно 0, появляется только после дефолта.
3. **`bureau_score_latest` (из `scoring_applications_full`) или самый свежий снимок из `scoring_bureau_snapshots` без фильтра `<= decision_date`:** содержит скор БКИ на май 2025 года, который уже упал **вследствие** неплатежей по нашему же кредиту!
4. **`worst_dpd_ever`, `current_dpd_bucket`, `outstanding_uzs` (из `scoring_bureau_loans`):** отражают максимальную просрочку и остаток долга в БКИ по состоянию на `2025-05-31`, а не на момент заявки.
5. **`status` в `scoring_card_disputes` (`resolved_client` / `resolved_merchant`) и `status` в `scoring_cards` (`blocked`):** исход расследования чарджбэка и блокировка карты происходят спустя недели/месяцы после `decision_date`.

In [9]:
# 5.1. Сборка витрины фичей строго на момент decision_date (Point-in-Time)
features_sql = """
WITH b_asof AS (
    -- Последний снимок БКИ строго НЕ ПОЗЖЕ decision_date (as-of join)
    SELECT DISTINCT ON (a.application_id)
        a.application_id,
        b.snapshot_date AS bureau_snapshot_date,
        (a.decision_date - b.snapshot_date) AS bureau_snapshot_age_days,
        b.bureau_score AS bureau_score_asof,
        b.existing_debt_uzs AS existing_debt_uzs_asof,
        b.active_loans_cnt AS active_loans_cnt_asof
    FROM scoring_applications a
    LEFT JOIN scoring_bureau_snapshots b
      ON a.client_id = b.client_id
     AND b.snapshot_date <= a.decision_date
    ORDER BY a.application_id, b.snapshot_date DESC
),
b_inq AS (
    -- Запросы в БКИ за 90 и 180 дней строго до decision_date
    SELECT 
        a.application_id,
        COUNT(i.inquiry_id) FILTER (
            WHERE i.inquiry_date <= a.decision_date 
              AND i.inquiry_date >= a.decision_date - INTERVAL '90 days'
        ) AS bureau_inq_90d_cnt,
        COUNT(i.inquiry_id) FILTER (
            WHERE i.inquiry_date <= a.decision_date 
              AND i.inquiry_date >= a.decision_date - INTERVAL '180 days'
        ) AS bureau_inq_180d_cnt
    FROM scoring_applications a
    LEFT JOIN scoring_bureau_inquiries i
      ON a.client_id = i.client_id
     AND i.inquiry_date <= a.decision_date
    GROUP BY 1
),
cards_pre AS (
    -- Карты клиента, выпущенные строго до decision_date (без поля status!)
    SELECT 
        a.application_id,
        COUNT(ca.card_id) AS cards_cnt_before,
        COALESCE(MAX(ca.credit_limit_uzs), 0) AS max_card_limit_uzs
    FROM scoring_applications a
    LEFT JOIN scoring_cards ca
      ON a.client_id = ca.customer_id
     AND ca.issue_date < a.decision_date
    GROUP BY 1
),
txns_pre AS (
    -- Транзакции по картам за 3 и 6 месяцев строго до decision_date
    SELECT 
        a.application_id,
        COUNT(t.txn_id) FILTER (WHERE t.txn_ts >= a.decision_date - INTERVAL '3 months') AS card_txns_3m_cnt,
        COUNT(t.txn_id) FILTER (WHERE t.txn_ts >= a.decision_date - INTERVAL '6 months') AS card_txns_6m_cnt,
        COALESCE(SUM(t.amount_uzs) FILTER (
            WHERE t.status = 'approved' AND t.txn_ts >= a.decision_date - INTERVAL '3 months'
        ), 0) AS card_turnover_3m_uzs,
        COALESCE(SUM(t.amount_uzs) FILTER (
            WHERE t.status = 'approved' AND t.txn_ts >= a.decision_date - INTERVAL '6 months'
        ), 0) AS card_turnover_6m_uzs,
        -- Входящая зарплата за 6 месяцев
        COALESCE(SUM(t.amount_uzs) FILTER (
            WHERE t.status = 'approved' AND t.txn_type = 'salary_in' AND t.txn_ts >= a.decision_date - INTERVAL '6 months'
        ), 0) / 6.0 AS salary_in_avg_m_6m_uzs,
        COUNT(DISTINCT DATE_TRUNC('month', t.txn_ts)) FILTER (
            WHERE t.status = 'approved' AND t.txn_type = 'salary_in' AND t.txn_ts >= a.decision_date - INTERVAL '6 months'
        ) AS salary_months_cnt_6m,
        -- Снятия наличных (cash_out) и расходы (purchase + cash_out + p2p_out) за 6 месяцев
        COALESCE(SUM(t.amount_uzs) FILTER (
            WHERE t.status = 'approved' AND t.txn_type = 'cash_out' AND t.txn_ts >= a.decision_date - INTERVAL '6 months'
        ), 0) AS cash_out_6m_uzs,
        COALESCE(SUM(t.amount_uzs) FILTER (
            WHERE t.status = 'approved' AND t.txn_type IN ('purchase', 'cash_out', 'p2p_out') AND t.txn_ts >= a.decision_date - INTERVAL '6 months'
        ), 0) AS total_out_6m_uzs,
        -- Деклайны за 6 месяцев
        COUNT(t.txn_id) FILTER (
            WHERE t.status = 'declined' AND t.txn_ts >= a.decision_date - INTERVAL '6 months'
        ) AS declined_txns_6m_cnt
    FROM scoring_applications a
    LEFT JOIN scoring_card_transactions t
      ON a.client_id = t.client_id
     AND t.txn_ts >= (a.decision_date - INTERVAL '6 months')
     AND t.txn_ts < a.decision_date
    GROUP BY 1
),
disputes_pre AS (
    -- Диспуты, открытые строго до decision_date (без итогового статуса разрешения!)
    SELECT 
        a.application_id,
        COUNT(d.dispute_id) AS disputes_cnt_before,
        COALESCE(SUM(d.dispute_amount_uzs), 0) AS disputes_amt_before_uzs
    FROM scoring_applications a
    JOIN scoring_card_transactions t ON a.client_id = t.client_id AND t.txn_ts < a.decision_date
    JOIN scoring_card_disputes d ON t.txn_id = d.txn_id AND d.opened_date < a.decision_date
    GROUP BY 1
)
SELECT 
    a.application_id,
    a.client_id,
    a.decision_date,
    a.final_decision,
    -- Анкетные фичи
    a.channel,
    a.purpose,
    a.requested_amount_uzs,
    a.term_months,
    a.declared_income_uzs,
    a.employment_months,
    CASE WHEN a.employment_months IS NULL THEN 1 ELSE 0 END AS is_emp_missing,
    ROUND((a.requested_amount_uzs::numeric / a.term_months) / NULLIF(a.declared_income_uzs, 0), 4) AS pti_simple_ratio,
    -- Профиль клиента (ETB vs NTB)
    CASE WHEN cu.customer_id IS NOT NULL AND cu.onboarding_date <= a.decision_date THEN 1 ELSE 0 END AS is_existing_customer,
    COALESCE(cu.customer_segment, 'NTB') AS customer_segment,
    EXTRACT(YEAR FROM AGE(a.decision_date, cu.birth_date))::int AS age_years,
    -- Фичи БКИ (as-of decision_date)
    b.bureau_score_asof,
    b.existing_debt_uzs_asof,
    b.active_loans_cnt_asof,
    b.bureau_snapshot_age_days,
    ROUND(b.existing_debt_uzs_asof::numeric / NULLIF(a.declared_income_uzs, 0), 4) AS debt_to_declared_income,
    bi.bureau_inq_90d_cnt,
    bi.bureau_inq_180d_cnt,
    -- Карточные фичи за 3-6 мес. до decision_date
    cp.cards_cnt_before,
    cp.max_card_limit_uzs,
    tx.card_txns_3m_cnt,
    tx.card_txns_6m_cnt,
    tx.card_turnover_3m_uzs,
    tx.card_turnover_6m_uzs,
    ROUND((tx.card_turnover_3m_uzs * 2.0) / NULLIF(tx.card_turnover_6m_uzs, 0), 4) AS card_turnover_3m_to_6m_ratio,
    ROUND(tx.salary_in_avg_m_6m_uzs, 0) AS salary_in_avg_m_6m_uzs,
    tx.salary_months_cnt_6m,
    ROUND(tx.salary_in_avg_m_6m_uzs / NULLIF(a.declared_income_uzs, 0), 4) AS salary_to_declared_ratio,
    CASE 
        WHEN tx.salary_months_cnt_6m = 0 THEN '0_no_salary_on_card'
        WHEN tx.salary_in_avg_m_6m_uzs >= a.declared_income_uzs * 0.7 THEN '1_verified (>=70%)'
        ELSE '2_overstated (<70%)'
    END AS income_verification_status,
    ROUND(tx.cash_out_6m_uzs::numeric / NULLIF(tx.total_out_6m_uzs, 0), 4) AS cash_out_share_6m,
    ROUND(tx.declined_txns_6m_cnt::numeric / NULLIF(tx.card_txns_6m_cnt, 0), 4) AS decline_rate_6m,
    COALESCE(dp.disputes_cnt_before, 0) AS disputes_cnt_before,
    COALESCE(dp.disputes_amt_before_uzs, 0) AS disputes_amt_before_uzs
FROM scoring_applications a
LEFT JOIN scoring_customers cu ON a.client_id = cu.customer_id AND cu.onboarding_date <= a.decision_date
LEFT JOIN b_asof b ON a.application_id = b.application_id
LEFT JOIN b_inq bi ON a.application_id = bi.application_id
LEFT JOIN cards_pre cp ON a.application_id = cp.application_id
LEFT JOIN txns_pre tx ON a.application_id = tx.application_id
LEFT JOIN disputes_pre dp ON a.application_id = dp.application_id
ORDER BY a.application_id
"""

df_features = q(features_sql)
df_modeling = df_features.merge(df_target[["application_id", "target"]], on="application_id", how="left")

print(f"Размер итогового датасета фичей: {df_features.shape[0]:,} строк x {df_features.shape[1]} колонок")
print(f"Из них размечено честным зрелым таргетом (df_modeling['target'].notna()): {df_modeling['target'].notna().sum():,} строк")
display(df_modeling.head(10))

Размер итогового датасета фичей: 18,000 строк x 37 колонок
Из них размечено честным зрелым таргетом (df_modeling['target'].notna()): 7,852 строк


,application_id,client_id,decision_date,final_decision,channel,purpose,requested_amount_uzs,term_months,declared_income_uzs,employment_months,is_emp_missing,pti_simple_ratio,is_existing_customer,customer_segment,age_years,bureau_score_asof,existing_debt_uzs_asof,active_loans_cnt_asof,bureau_snapshot_age_days,debt_to_declared_income,bureau_inq_90d_cnt,bureau_inq_180d_cnt,cards_cnt_before,max_card_limit_uzs,card_txns_3m_cnt,card_txns_6m_cnt,card_turnover_3m_uzs,card_turnover_6m_uzs,card_turnover_3m_to_6m_ratio,salary_in_avg_m_6m_uzs,salary_months_cnt_6m,salary_to_declared_ratio,income_verification_status,cash_out_share_6m,decline_rate_6m,disputes_cnt_before,disputes_amt_before_uzs,target
0,1,529,2024-08-27,approved,mobile_app,education,20800000,18,6310000,204.0,0,0.1831,1,payroll,50.0,644.0,16000000.0,3.0,8.0,2.5357,0,1,2,0,20,40,25863961.0,49053990.0,1.0545,6540923.0,6,1.0366,1_verified (>=70%),0.0673,0.0750,0,0.0,0.0
1,2,920,2023-10-19,approved,branch,education,15300000,12,11120000,0.0,0,0.1147,1,payroll,32.0,745.0,9000000.0,3.0,6.0,0.8094,0,0,1,0,10,32,2270559.0,6734432.0,0.6743,0.0,0,0.0000,0_no_salary_on_card,0.1866,0.0313,0,0.0,0.0
2,3,7509,2023-07-01,approved,branch,education,3900000,12,4140000,92.0,0,0.0785,1,mass,56.0,725.0,19000000.0,2.0,46.0,4.5894,0,0,2,0,0,0,0.0,0.0,NaN,0.0,0,0.0000,0_no_salary_on_card,NaN,NaN,0,0.0,0.0
3,4,23494,2024-12-04,rejected,agent,medical,15500000,9,4340000,155.0,0,0.3968,0,NTB,NaN,527.0,20000000.0,1.0,149.0,4.6083,1,1,0,0,0,0,0.0,0.0,NaN,0.0,0,0.0000,0_no_salary_on_card,NaN,NaN,0,0.0,NaN
4,5,29522,2023-07-13,rejected,branch,consumer,14800000,18,3120000,57.0,0,0.2635,0,NTB,NaN,NaN,NaN,NaN,NaN,NaN,0,2,0,0,0,0,0.0,0.0,NaN,0.0,0,0.0000,0_no_salary_on_card,NaN,NaN,0,0.0,NaN
5,6,4526,2025-01-16,rejected,branch,education,2200000,6,17300000,124.0,0,0.0212,1,affluent,40.0,677.0,11000000.0,2.0,18.0,0.6358,0,0,2,0,20,46,55743902.0,112002233.0,0.9954,14143469.0,6,0.8175,1_verified (>=70%),0.2413,0.0000,1,1636667.0,NaN
6,7,4522,2024-06-17,approved,branch,repair,6100000,9,4500000,117.0,0,0.1506,1,mass,56.0,700.0,0.0,0.0,60.0,0.0000,0,0,1,10000000,29,46,21739144.0,40072612.0,1.0850,4982516.0,6,1.1072,1_verified (>=70%),0.0499,0.0435,0,0.0,1.0
7,8,22551,2024-11-24,rejected,branch,repair,9100000,12,4950000,217.0,0,0.1532,0,NTB,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0.0,0.0,NaN,0.0,0,0.0000,0_no_salary_on_card,NaN,NaN,0,0.0,NaN
8,9,2271,2025-04-06,rejected,agent,auto,33500000,12,24090000,143.0,0,0.1159,1,mass,56.0,703.0,2000000.0,1.0,20.0,0.0830,0,0,2,8000000,0,0,0.0,0.0,NaN,0.0,0,0.0000,0_no_salary_on_card,NaN,NaN,0,0.0,NaN
9,10,7478,2025-02-28,approved,agent,auto,2500000,24,14620000,177.0,0,0.0071,1,premium,47.0,698.0,7000000.0,3.0,65.0,0.4788,0,0,2,0,25,40,35593557.0,68284869.0,1.0425,9884256.0,6,0.6761,2_overstated (<70%),0.2725,0.0000,0,0.0,NaN


In [10]:
# 5.2. Проверка: подтверждается ли заявленный доход зарплатными зачислениями и как это влияет на дефолт?
print("--- 1. Подтверждение заявленного дохода зарплатными зачислениями (salary_in за 6 мес. до заявки) ---")
inc_ver_summary = (
    df_modeling.groupby(["is_existing_customer", "income_verification_status"], dropna=False)
    .agg(
        total_apps=("application_id", "count"),
        matured_loans=("target", lambda s: s.notna().sum()),
        avg_declared_mln=("declared_income_uzs", lambda s: round(s.mean() / 1e6, 2)),
        avg_card_salary_mln=("salary_in_avg_m_6m_uzs", lambda s: round(s.mean() / 1e6, 2)),
        median_sal_to_decl=("salary_to_declared_ratio", lambda s: round(s.median(), 2)),
        bad_rate_pct=("target", lambda s: round(s.mean() * 100, 2)),
    )
    .reset_index()
)
display(inc_ver_summary)

# 5.3. Демонстрация опасности колонок-ловушек: сравнение честного as-of бюро-скора с колонками из будущего
print("--- 2. Сравнение честных фичей и колонок-ловушек (корреляция с таргетом на зрелых кредитах) ---")
leak_check_sql = """
SELECT 
    a.application_id,
    f.bureau_score_latest AS trap_bureau_score_latest,
    f.dpd_max AS trap_dpd_max,
    f.collection_calls_cnt AS trap_collection_calls_cnt,
    bl.max_worst_dpd_ever AS trap_bureau_worst_dpd_ever
FROM scoring_applications a
JOIN scoring_applications_full f ON a.application_id = f.application_id
LEFT JOIN (
    SELECT client_id, MAX(worst_dpd_ever) AS max_worst_dpd_ever
    FROM scoring_bureau_loans
    GROUP BY 1
) bl ON a.client_id = bl.client_id
"""
df_leaks = q(leak_check_sql)
df_leak_cmp = df_modeling[df_modeling["target"].notna()].merge(df_leaks, on="application_id", how="left")

corr_cols = [
    "bureau_score_asof",
    "decline_rate_6m",
    "salary_to_declared_ratio",
    "trap_bureau_score_latest",
    "trap_bureau_worst_dpd_ever",
    "trap_collection_calls_cnt",
    "trap_dpd_max",
]
corr_df = pd.DataFrame({
    "feature": corr_cols,
    "type": [
        "Честная фича (до decision_date)",
        "Честная фича (до decision_date)",
        "Честная фича (до decision_date)",
        "ЛОВУШКА (из будущего: scoring_applications_full)",
        "ЛОВУШКА (из будущего: scoring_bureau_loans)",
        "ЛОВУШКА (из будущего: scoring_applications_full)",
        "ЛОВУШКА (прямой лик таргета: scoring_applications_full)",
    ],
    "corr_with_target": [round(df_leak_cmp[c].corr(df_leak_cmp["target"]), 4) for c in corr_cols],
})
display(corr_df)

--- 1. Подтверждение заявленного дохода зарплатными зачислениями (salary_in за 6 мес. до заявки) ---


,is_existing_customer,income_verification_status,total_apps,matured_loans,avg_declared_mln,avg_card_salary_mln,median_sal_to_decl,bad_rate_pct
0,0,0_no_salary_on_card,7137,2326,7.18,0.00,0.00,25.32
1,1,0_no_salary_on_card,5941,3022,8.72,0.00,0.00,12.18
2,1,1_verified (>=70%),3662,1847,10.30,8.89,0.86,12.56
3,1,2_overstated (<70%),1260,657,13.56,8.89,0.66,14.92


--- 2. Сравнение честных фичей и колонок-ловушек (корреляция с таргетом на зрелых кредитах) ---


,feature,type,corr_with_target
0,bureau_score_asof,Честная фича (до decision_date),-0.1043
1,decline_rate_6m,Честная фича (до decision_date),0.0516
2,salary_to_declared_ratio,Честная фича (до decision_date),-0.0598
3,trap_bureau_score_latest,ЛОВУШКА (из будущего: scoring_applications_full),-0.3831
4,trap_bureau_worst_dpd_ever,ЛОВУШКА (из будущего: scoring_bureau_loans),0.2903
5,trap_collection_calls_cnt,ЛОВУШКА (из будущего: scoring_applications_full),0.8877
6,trap_dpd_max,ЛОВУШКА (прямой лик таргета: scoring_applicati...,0.8315


## Шаг 6. Схема проверки

Модель обучать не нужно. Опишите:

- разбиение по времени (out-of-time), а не случайное, с границами периодов;
- какие метрики смотрите и почему (ранжирование отдельно, экономика решения отдельно);
- цену двух ошибок (одобрили bad, отклонили good) в деньгах на примере среднего договора.

## Как сдавать

Решайте в этом ноутбуке. Ответы (числа и текст, включая логику вашего таргета)
впишите в форму задания на платформе.

In [11]:
# 6.1. Расчёт объёмов выборок при разбиении по времени (Out-of-Time)
print("--- 1. Разбиение выборок по времени (Train / In-Time Val / Out-of-Time Test / PSI Monitoring) ---")
df_modeling["dec_month"] = pd.to_datetime(df_modeling["decision_date"]).dt.to_period("M")

def assign_split(dt):
    d = pd.to_datetime(dt)
    if d <= pd.Timestamp("2024-05-31"):
        return "1_Train (2023-06 .. 2024-05, до скачка)"
    elif d <= pd.Timestamp("2024-07-31"):
        return "2_Val_OOT1 (2024-06 .. 2024-07, начало скачка)"
    elif d <= pd.Timestamp("2024-10-31"):
        return "3_Test_OOT2 (2024-08 .. 2024-10, зрелый OOT)"
    else:
        return "4_Unmatured_PSI (2024-11 .. 2025-04, мониторинг PSI / FPD)"

df_modeling["split_group"] = df_modeling["decision_date"].apply(assign_split)

split_summary = (
    df_modeling.groupby("split_group")
    .agg(
        total_apps=("application_id", "count"),
        approved_apps=("final_decision", lambda s: (s == "approved").sum()),
        matured_with_target=("target", lambda s: s.notna().sum()),
        bad_cnt=("target", lambda s: (s == 1).sum()),
        bad_rate_pct=("target", lambda s: round(s.mean() * 100, 2)),
    )
    .reset_index()
)
display(split_summary)

# 6.2. Расчёт цены двух ошибок (FN и FP) в деньгах на примере среднего договора из базы
print("--- 2. Экономика среднего договора и фактические потери на Bad vs доход на Good (зрелые когорты) ---")
econ_sql = """
WITH contract_cashflows AS (
    SELECT 
        c.contract_id,
        c.issue_date,
        c.issued_amount_uzs,
        c.term_months,
        c.annual_rate_pct,
        c.monthly_payment_uzs,
        (c.monthly_payment_uzs * c.term_months - c.issued_amount_uzs) AS planned_interest_uzs,
        COALESCE(SUM(p.paid_amount_uzs), 0) AS total_paid_uzs,
        MAX(CASE 
            WHEN p.paid_date IS NOT NULL THEN GREATEST(0, (p.paid_date - p.due_date))
            WHEN p.due_date <= DATE '2025-05-31' THEN (DATE '2025-05-31' - p.due_date)
            ELSE 0
        END) FILTER (WHERE p.installment_no <= 6) AS dpd_6m
    FROM scoring_contracts c
    LEFT JOIN scoring_payments p ON c.contract_id = p.contract_id
    WHERE c.issue_date <= DATE '2024-05-31'
    GROUP BY 1, 2, 3, 4, 5, 6
)
SELECT 
    CASE WHEN dpd_6m >= 60 THEN 'Bad (target = 1, DPD 60+)' ELSE 'Good (target = 0)' END AS client_group,
    COUNT(*) AS contracts_cnt,
    ROUND(AVG(issued_amount_uzs), 0) AS avg_issued_uzs,
    ROUND(AVG(term_months), 1) AS avg_term_months,
    ROUND(AVG(annual_rate_pct), 2) AS avg_rate_pct,
    ROUND(AVG(monthly_payment_uzs), 0) AS avg_monthly_pmt_uzs,
    ROUND(AVG(planned_interest_uzs), 0) AS avg_planned_interest_uzs,
    ROUND(AVG(total_paid_uzs), 0) AS avg_actual_paid_uzs,
    ROUND(AVG(total_paid_uzs - issued_amount_uzs), 0) AS avg_net_cash_margin_uzs,
    ROUND(AVG(GREATEST(0, issued_amount_uzs - total_paid_uzs)), 0) AS avg_unrecovered_principal_uzs,
    ROUND(AVG(GREATEST(0, issued_amount_uzs - total_paid_uzs)) / AVG(issued_amount_uzs) * 100, 1) AS empirical_lgd_pct
FROM contract_cashflows
GROUP BY 1
ORDER BY 1
"""
df_econ = q(econ_sql)
display(df_econ)

--- 1. Разбиение выборок по времени (Train / In-Time Val / Out-of-Time Test / PSI Monitoring) ---


,split_group,total_apps,approved_apps,matured_with_target,bad_cnt,bad_rate_pct
0,"1_Train (2023-06 .. 2024-05, до скачка)",9347,5917,5917,865,14.62
1,"2_Val_OOT1 (2024-06 .. 2024-07, начало скачка)",1554,1000,1000,210,21.00
2,"3_Test_OOT2 (2024-08 .. 2024-10, зрелый OOT)",2312,1450,935,212,22.67
3,"4_Unmatured_PSI (2024-11 .. 2025-04, мониторин...",4787,2910,0,0,NaN


--- 2. Экономика среднего договора и фактические потери на Bad vs доход на Good (зрелые когорты) ---


,client_group,contracts_cnt,avg_issued_uzs,avg_term_months,avg_rate_pct,avg_monthly_pmt_uzs,avg_planned_interest_uzs,avg_actual_paid_uzs,avg_net_cash_margin_uzs,avg_unrecovered_principal_uzs,empirical_lgd_pct
0,"Bad (target = 1, DPD 60+)",858,13264103.0,13.4,31.12,1385416.0,2646097.0,4581344.0,-8682758.0,9089820.0,68.5
1,Good (target = 0),5016,11694976.0,13.6,31.06,1200445.0,2385601.0,13106808.0,1411832.0,226966.0,1.9


### Описание схемы проверки, метрик и экономики ошибок (Шаг 6)

#### 1. Разбиение по времени (Out-of-Time Validation), а не случайное
Случайный сплит (`train_test_split(shuffle=True)`) в кредитном скоринге **недопустим**, так как:
1. Он перемешивает макроэкономические и продуктовые сдвиги (в нашем случае — июньский скачок просрочки 2024 года попадёт одновременно и в `train`, и в `test`, завышая метрику).
2. Один и тот же клиент (`client_id`) с повторными заявками может попасть и в обучение, и в тест.
3. Между выдачей кредита и созреванием 6-месячного таргета проходит минимум **6–7 месяцев** (*maturity lag*). В реальной жизни модель обучается только на тех когортах, где таргет уже созрел, а применяется к новым заявкам.

**Границы периодов (`decision_date`):**
- **Train (`2023-06-01` – `2024-05-31`, 12 месяцев):** **5 917 зрелых выданных договоров** (9 347 всех заявок), доля `bad = 14.62%`. На этом периоде модель обучается (внутри `Train` выделяется 15% In-Time Validation с группировкой по `client_id` для тюнинга гиперпараметров и ранняя остановка).
- **Val OOT-1 (`2024-06-01` – `2024-07-31`, 2 месяца):** **1 000 зрелых договоров** (1 554 заявки), доля `bad = 21.00%`. Период начала роста просрочки — используется для калибровки вероятностей (Platt / Isotonic) и выбора порога отсечения (`cutoff`).
- **Test OOT-2 (`2024-08-01` – `2024-10-31`, 3 месяца):** **1 450 зрелых договоров** (2 312 заявок), доля `bad = 24.55%`. Финальный слепой Out-of-Time тест, проверяющий способность модели удерживать риск после структурного сдвига портфеля.
- **Unmatured / Monitoring (`2024-11-01` – `2025-04-30`, 6 месяцев):** **4 787 заявок** (2 910 выданных кредитов с незакрытым 6-месячным окном). В обучение и расчёт 6-месячного ROC-AUC **не входят**, но используются для:
  - проверки стабильности распределения признаков и скорбалла (**PSI — Population Stability Index** на всех входящих заявках, включая `rejected`);
  - быстрой проверки по короткому раннему таргету **FPD30 / SPD30** (`DPD 30+` на первых 1–3 платежах для выдач ноября 2024 – января 2025).

---

#### 2. Какие метрики смотрим и почему

**А. Метрики качества ранжирования (отдельно от порога):**
1. **ROC-AUC и коэффициент Gini ($\text{Gini} = 2 \cdot \text{ROC-AUC} - 1$):** главная метрика разделяющей способности скоркарты во всём диапазоне баллов. Оцениваем отдельно на `In-Time`, `OOT-1` и `OOT-2`, а также в разрезе сегментов **ETB** (с карточными фичами) и **NTB** (без карточных фичей), и сравниваем с baseline старой модели `score_v1`.
2. **PR-AUC (Average Precision) и Lift@K% (например, Lift в топ-10% и топ-20% самых рискованных заявок):** показывают, насколько эффективно модель концентрирует `bad`-клиентов в хвосте распределения при изменении базового Bad Rate с 14.6% до 24.6%.
3. **KS-статистика (Kolmogorov-Smirnov):** максимальное расхождение кумулятивных распределений `good` и `bad` — помогает ориентироваться при выборе рабочего порога.
4. **PSI (Population Stability Index) и калибровочная кривая (Brier Score / Expected Calibration Error):** проверяют, не «поплыл» ли скор во времени и соответствует ли предсказанная вероятность дефолта ($\text{PD}$) фактической доле дефолтов в каждом дециле.

**Б. Метрики экономики бизнес-решения (при фиксированном пороге `cutoff`):**
1. **Approval Rate (доля одобрения, %)** и **Bad Rate в одобренном портфеле (%)** — ключевой компромисс для бизнеса (например: снизить Bad Rate при том же Approval Rate $\approx 62.6\%$, либо удержать Bad Rate $\le 14\%$ при максимальном одобрении).
2. **Expected Profit / NPV портфеля (в сумах на 1 000 входящих заявок):** суммарный финансовый результат с учётом матрицы цен ошибок.

---

#### 3. Цена двух ошибок в деньгах на примере среднего договора

По данным `scoring_contracts` средний выданный договор имеет параметры:
- **Средняя сумма выдачи ($\text{EAD}$):** **`11 658 766 сум`** (медиана `9 000 000 сум`);
- **Средний срок:** **`13.55 мес.`**; **средняя ставка:** **`31.03% годовых`**; **средний ежемесячный платёж:** **`1 207 298 сум`**.

Сопоставление фактических денежных потоков (`scoring_payments`) на зрелых когортах даёт точную стоимость двух ошибок классификации:

| Тип ошибки модели | Что произошло в бизнесе | Расчёт на среднем договоре из базы | Цена одной ошибки в деньгах |
|---|---|---|---|
| **Ошибка I рода (False Negative — одобрили `bad`)** | Модель предсказала `good`, банк выдал кредит, но клиент ушёл в дефолт (`DPD 60+`) | Средний `bad`-клиент получает **`13 264 103 сум`**, но за время жизни кредита успевает выплатить лишь **`4 581 344 сум`** (около 3.5 платежей до остановки оплат). Прямой невозврат тела кредита составляет **`8 682 758 сум`** (а средний невозвращённый остаток основного долга с учётом LGD $\approx 68.5\%$ равен **`9 089 820 сум`** + расходы на коллекшен). На стандартный средний чек `11.66 млн сум` при LGD = 68.5% чистый убыток равен $11.66 \times 0.685 \approx 7.99\text{ млн сум}$. | **~8.68 млн сум** прямого убытка (или **~8.0 млн сум** в пересчёте на средний чек `11.66 млн сум`) |
| **Ошибка II рода (False Positive — отклонили `good`)** | Модель предсказала `bad`, банк отказал надёжному клиенту | Банк не теряет тело кредита, но теряет упущенный процентный доход (маржу). Плановый процентный доход по среднему договору ($	ext{monthly\_payment} \times 	ext{term} - 	ext{issued\_amount}$) составляет **`2 347 415 сум`** (а за вычетом стоимости фондирования банка чистая процентная маржа составляет порядка **1.41 – 2.35 млн сум**). | **~2.35 млн сум** упущенного процентного дохода (или **~1.41 млн сум** фактического чистого денежного потока) |

**Вывод для порога отсечения (`cutoff`):**
- Одобрить одного `bad`-клиента (**~8.68 млн сум** убытка) в **3.7 – 6.1 раза дороже**, чем отказать одному `good`-клиенту (**~1.41 – 2.35 млн сум** упущенной выгоды).
- Следовательно, оптимальный экономический порог вероятности дефолта ($\text{PD}_{\text{cutoff}}$), при котором выгодно одобрять заявку, определяется условием безубыточности:
  $$\text{PD} \cdot C_{\text{FN}} \le (1 - \text{PD}) \cdot C_{\text{FP}} \quad \Longrightarrow \quad \text{PD}_{\text{cutoff}} = \frac{C_{\text{FP}}}{C_{\text{FN}} + C_{\text{FP}}} = \frac{2.35}{8.68 + 2.35} \approx 21.3\% \quad (\text{или } \approx 14.0\% \text{ при чистой марже } 1.41\text{ млн сум}).$$
- Все заявки с прогнозируемой вероятностью дефолта $\text{PD} > 21\%$ экономически убыточны для одобрения на стандартных условиях.